# Train / Val / Test Splits — Nivel de Paciente

Define la partición de datos **a nivel de `subject_id`** para evitar data leakage:  
un mismo paciente no puede aparecer en train y test a la vez.

**Proporciones**: 70% train · 15% val · 15% test  
**Estratificación**: por label de sepsis, para mantener la prevalencia en cada split.  
**Semilla fija**: reproducibilidad garantizada.

**Salida**: `data/processed/splits.parquet`  
Columnas: `subject_id`, `stay_id`, `sepsis`, `split` (`train`/`val`/`test`)

In [1]:
import polars as pl
import numpy as np
from pathlib import Path

COHORT_PATH = Path("../data/processed/cohort.parquet")
OUT_DIR     = Path("../data/processed")

SEED        = 42
TRAIN_FRAC  = 0.70
VAL_FRAC    = 0.15
# TEST_FRAC  = 0.15  (el resto)

print(f"Seed: {SEED}  |  Train: {TRAIN_FRAC:.0%}  Val: {VAL_FRAC:.0%}  Test: {1-TRAIN_FRAC-VAL_FRAC:.0%}")

Seed: 42  |  Train: 70%  Val: 15%  Test: 15%


## PASO 1 — Cargar cohorte y agregar por paciente

La unidad de split es el **paciente** (`subject_id`).  
Si un paciente tiene varios stays, todos van al mismo split.

In [2]:
cohort = pl.read_parquet(COHORT_PATH).select([
    "subject_id", "stay_id", "sepsis"
])

# Un paciente es "caso de sepsis" si tiene al menos un stay con sepsis
patients = (
    cohort
    .group_by("subject_id")
    .agg(
        pl.col("sepsis").max().alias("sepsis_patient"),  # 1 si algún stay es séptico
        pl.col("stay_id").count().alias("n_stays"),
    )
)

n_patients = len(patients)
n_sepsis_patients = patients["sepsis_patient"].sum()
print(f"Pacientes totales:       {n_patients:,}")
print(f"Con sepsis (≥1 stay):    {n_sepsis_patients:,}  ({100*n_sepsis_patients/n_patients:.1f}%)")
print(f"Sin sepsis:              {n_patients - n_sepsis_patients:,}")
print(f"\nDistribución de stays por paciente:")
print(patients["n_stays"].describe())

Pacientes totales:       54,551
Con sepsis (≥1 stay):    13,550  (24.8%)
Sin sepsis:              41,001

Distribución de stays por paciente:
shape: (9, 2)
┌────────────┬──────────┐
│ statistic  ┆ value    │
│ ---        ┆ ---      │
│ str        ┆ f64      │
╞════════════╪══════════╡
│ count      ┆ 54551.0  │
│ null_count ┆ 0.0      │
│ mean       ┆ 1.371726 │
│ std        ┆ 1.013368 │
│ min        ┆ 1.0      │
│ 25%        ┆ 1.0      │
│ 50%        ┆ 1.0      │
│ 75%        ┆ 1.0      │
│ max        ┆ 37.0     │
└────────────┴──────────┘


## PASO 2 — Split estratificado por sepsis a nivel de paciente

In [3]:
rng = np.random.default_rng(SEED)

def stratified_split(df: pl.DataFrame, label_col: str, train_frac: float, val_frac: float, rng):
    """Split estratificado por label. Devuelve columna 'split'."""
    splits = []
    for label_val in [0, 1]:
        ids = df.filter(pl.col(label_col) == label_val)["subject_id"].to_numpy()
        rng.shuffle(ids)
        n = len(ids)
        n_train = int(n * train_frac)
        n_val   = int(n * val_frac)
        assignment = (
            ["train"] * n_train +
            ["val"]   * n_val +
            ["test"]  * (n - n_train - n_val)
        )
        splits.append(pl.DataFrame({"subject_id": ids, "split": assignment}))
    return pl.concat(splits)

patient_splits = stratified_split(patients, "sepsis_patient", TRAIN_FRAC, VAL_FRAC, rng)

print("Distribución de pacientes por split:")
print(
    patient_splits
    .join(patients.select(["subject_id", "sepsis_patient"]), on="subject_id", how="left")
    .group_by(["split", "sepsis_patient"])
    .agg(pl.len().alias("n_pacientes"))
    .sort(["split", "sepsis_patient"])
)

Distribución de pacientes por split:
shape: (6, 3)
┌───────┬────────────────┬─────────────┐
│ split ┆ sepsis_patient ┆ n_pacientes │
│ ---   ┆ ---            ┆ ---         │
│ str   ┆ i8             ┆ u32         │
╞═══════╪════════════════╪═════════════╡
│ test  ┆ 0              ┆ 6151        │
│ test  ┆ 1              ┆ 2033        │
│ train ┆ 0              ┆ 28700       │
│ train ┆ 1              ┆ 9485        │
│ val   ┆ 0              ┆ 6150        │
│ val   ┆ 1              ┆ 2032        │
└───────┴────────────────┴─────────────┘


## PASO 3 — Propagar split a nivel de stay

In [4]:
splits = (
    cohort
    .join(patient_splits, on="subject_id", how="left")
    .select(["subject_id", "stay_id", "sepsis", "split"])
)

print("Distribución de stays por split:")
summary = (
    splits
    .group_by("split")
    .agg([
        pl.len().alias("n_stays"),
        pl.col("sepsis").sum().alias("n_sepsis"),
        (pl.col("sepsis").sum() / pl.len() * 100).round(1).alias("prevalencia_pct"),
        pl.col("subject_id").n_unique().alias("n_pacientes"),
    ])
    .sort("split")
)
print(summary)

# Sanity check: no leakage
train_patients = set(splits.filter(pl.col("split") == "train")["subject_id"].to_list())
val_patients   = set(splits.filter(pl.col("split") == "val")["subject_id"].to_list())
test_patients  = set(splits.filter(pl.col("split") == "test")["subject_id"].to_list())

leakage_tv = len(train_patients & val_patients)
leakage_tt = len(train_patients & test_patients)
leakage_vt = len(val_patients   & test_patients)

print(f"\n✓ Leakage train∩val:   {leakage_tv} pacientes")
print(f"✓ Leakage train∩test:  {leakage_tt} pacientes")
print(f"✓ Leakage val∩test:    {leakage_vt} pacientes")
assert leakage_tv == leakage_tt == leakage_vt == 0, "LEAKAGE DETECTADO"

Distribución de stays por split:
shape: (3, 5)
┌───────┬─────────┬──────────┬─────────────────┬─────────────┐
│ split ┆ n_stays ┆ n_sepsis ┆ prevalencia_pct ┆ n_pacientes │
│ ---   ┆ ---     ┆ ---      ┆ ---             ┆ ---         │
│ str   ┆ u32     ┆ i64      ┆ f64             ┆ u32         │
╞═══════╪═════════╪══════════╪═════════════════╪═════════════╡
│ test  ┆ 11193   ┆ 2589     ┆ 23.1            ┆ 8184        │
│ train ┆ 52383   ┆ 12005    ┆ 22.9            ┆ 38185       │
│ val   ┆ 11253   ┆ 2595     ┆ 23.1            ┆ 8182        │
└───────┴─────────┴──────────┴─────────────────┴─────────────┘

✓ Leakage train∩val:   0 pacientes
✓ Leakage train∩test:  0 pacientes
✓ Leakage val∩test:    0 pacientes


## PASO 4 — Guardar

In [5]:
output_path = OUT_DIR / "splits.parquet"
splits.write_parquet(output_path)

print(f"Guardado en: {output_path}")
print(f"Shape: {splits.shape}")
splits.head(8)

Guardado en: ../data/processed/splits.parquet
Shape: (74829, 4)


[Vista previa de registros individuales omitida — DUA de PhysioNet: los datos de MIMIC-IV no se redistribuyen]
